# Task Execution
This notebook demonstrates how to use Celery tasks to process astronomical images.

**Note:** This system only processes FITS (.fits) and NumPy (.npy) files. Other file formats are not supported.

## Processing a Single Image
The following example demonstrates how to process a single image using `process_image_task`.
The image path is relative to `IMAGE_BASE_PATH`.

Note: The `instrument_name` parameter allows you to specify a custom instrument configuration. For more details on instrument configurations, see the [Instrument Configuration Notebook](./2_Instrument_Configuration_Notebook.ipynb).

In [ ]:
from gpuphot_worker.tasks import process_image_task

# Example 1: Process a single image with the default configuration
image_path = 'prered/example_image.fits'  # Relative to IMAGE_BASE_PATH
result = process_image_task.delay(image_path)  # No instrument_name provided
print(f'Task ID for processing {image_path}: {result.id}')

# Example 2: Process a single image with a custom instrument configuration
instrument_name = 'my_instrument'  # Use a custom instrument configuration
result = process_image_task.delay(image_path, instrument_name=instrument_name)
print(f'Task ID for processing {image_path} with {instrument_name}: {result.id}')

## Processing a Directory of Images
The following examples demonstrate how to use `process_directory_task` to process a directory of images.
The directory path is relative to `IMAGE_BASE_PATH`.

**Parameters:**
- `path`: Subdirectory to search for images. If `None`, searches in the base image path.
- `filename`: Specific filename or pattern to match. Supports partial matches and wildcards.
- `instrument_name`: Overrides the default instrument name. See [Instrument Configuration](./2_Instrument_Configuration_Notebook.ipynb) for details.
- `exclude_pattern`: Regular expression pattern to exclude certain filenames.
- `reprocess`: If `True`, processes all found images; if `False`, skips images already processed.

In [ ]:
from gpuphot_worker.tasks import process_directory_task

# Example 1: Process all NPY files in the base directory with the default configuration
result = process_directory_task.delay(filename='*.npy')  # No path, no instrument_name
print(f'Task ID for processing all NPY files: {result.id}')

# Example 2: Process all FITS files in a specific subdirectory with the default configuration
result = process_directory_task.delay(path='prered', filename='*.fits')  # No instrument_name
print(f'Task ID for processing FITS files in "prered": {result.id}')

# Example 3: Process files matching a specific pattern in a subdirectory with a custom configuration
result = process_directory_task.delay(path='prered', filename='*QHY411*.fits', instrument_name='QHY411MERIS')
print(f'Task ID for processing QHY411 files in "prered": {result.id}')

# Example 4: Process files matching a specific pattern in the base directory with a custom configuration
result = process_directory_task.delay(filename='*iKon936*.fits', instrument_name='iKon936')
print(f'Task ID for processing iKon936 files: {result.id}')

# Example 5: Process files in a subdirectory, excluding files matching a pattern
result = process_directory_task.delay(path='prered', filename='*.fits', exclude_pattern='*bad*')
print(f'Task ID for processing FITS files in "prered" (excluding "bad" files): {result.id}')

# Example 6: Process files in a subdirectory without reprocessing already processed files
result = process_directory_task.delay(path='prered', filename='*.fits', reprocess=False)
print(f'Task ID for processing FITS files in "prered" (no reprocessing): {result.id}')

# Example 7: Process files in a subdirectory with a custom configuration and exclude pattern
result = process_directory_task.delay(path='prered', filename='*.fits', instrument_name='iKon936', exclude_pattern='*bad*')
print(f'Task ID for processing FITS files in "prered" with iKon936 config (excluding "bad" files): {result.id}')

# Example 8: Process files in a subdirectory with a custom configuration and force reprocessing
result = process_directory_task.delay(path='prered', filename='*.fits', instrument_name='iKon936', reprocess=True)
print(f'Task ID for processing FITS files in "prered" with iKon936 config (force reprocessing): {result.id}')

# Example 9: Process files in the base directory with a custom configuration and exclude pattern
result = process_directory_task.delay(filename='*.fits', instrument_name='iKon936', exclude_pattern='*bad*')
print(f'Task ID for processing FITS files with iKon936 config (excluding "bad" files): {result.id}')

# Example 10: Process files in the base directory with a custom configuration and force reprocessing
result = process_directory_task.delay(filename='*.fits', instrument_name='iKon936', reprocess=True)
print(f'Task ID for processing FITS files with iKon936 config (force reprocessing): {result.id}')

## Checking Task Status
After submitting a task, you can check its status using the task ID.

In [ ]:
from celery.result import AsyncResult

# Example: Check the status of a task
task_id = result.id  # Replace with your task ID
task_result = AsyncResult(task_id)

print(f'Task status: {task_result.status}')
print(f'Task result: {task_result.result}')

## Sending Tasks Directly with Celery App
In addition to using the predefined task functions (`process_image_task` and `process_directory_task`), you can send tasks directly using the Celery application (`app`). This approach provides more flexibility and control over task submission.

### Why Use `send_task`?
Using `send_task` allows you to integrate task submission directly into your own scripts or systems. For example, if you have a system for capturing astronomical images, you can automatically send a task to process each image as soon as it is saved. If the images are saved within the `IMAGE_PATH` directory, you can use the relative path to the image as the argument for the task.

### Example: Sending a Task with `send_task`
The following example demonstrates how to send a task directly using the `send_task` method. This method requires the name of the task and the arguments to pass to it.

In [ ]:
from gpuphot_worker.worker_app import app

# Example: Send a task to process a single image
task_name = 'gpuphot_worker.tasks.process_image_task'  # Name of the task
image_path = 'prered/example_image.fits'  # Relative to IMAGE_PATH
instrument_name = 'my_instrument'  # Optional: Custom instrument configuration

# Send the task
result = app.send_task(task_name, args=[image_path], kwargs={'instrument_name': instrument_name})
print(f'Task ID for processing {image_path} with {instrument_name}: {result.id}')

# Check the task status
from celery.result import AsyncResult
task_result = AsyncResult(result.id)
print(f'Task status: {task_result.status}')
print(f'Task result: {task_result.result}')

## Next Steps
If you need to revisit or modify instrument configurations, go back to the previous notebook:
- [2. Instrument Configuration](./2_Instrument_Configuration_Notebook.ipynb)

To explore the database and query photometric data, proceed to the next notebook:
- [4. Database Query](./4_Database_Query_Notebook.ipynb)